# 04 — Temporal-sparse PGD (Tier A) — Person 1, due Oct 23

Same PGD, same probes, same ε grid; all pixels but only k frames. Two frame-selection rules:
`random_k` (k frames uniformly at random, seeded per sample — mirrors He et al.) and `topk_grad`
(k frames with the largest |∂margin/∂x| at the clean input). k ∈ {1, 3, 5}. Selected indices are logged
per row (`frames_selected`). Output `results/raw/temporal_masked_results.csv`.

24 sweep points (2 rules × 3 k × 4 ε) — the largest Tier A job. Order below runs every (rule, k) at ε = 8/255
first, so a partial run already gives the comparison at the main operating point.

**Kaggle inputs:** Notebook 01's saved output (packed `CASIA-B-pkl`) · GPU T4 · Internet on ·
secret `GITHUB_TOKEN` (Contents read + write). Results are pushed to GitHub after every sweep point, so a
disconnect loses at most one probe; re-running resumes (same `run_id`).

**Prerequisite:** Notebook 02 has pushed `results/processed/subset_attack.csv`.
**Done when:** all 24 points done on the subset. **CHECKPOINT Oct 23:** all Tier A CSVs on identical samples.

In [ ]:
# --- Paths per platform. Edit ONLY this cell. -------------------------------------------
import os, sys, subprocess
from pathlib import Path

PLATFORM = "kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else \
           "colab" if "google.colab" in sys.modules else "local"
PATHS = {
    # Kaggle: team repo cloned into /kaggle/working; data + checkpoints from the private team Dataset.
    "kaggle": dict(repo="/kaggle/working/cross_attack_gait",
                   opengait="/kaggle/working/OpenGait",
                   dataset_root="/kaggle/input/casia-b-opengait/CASIA-B-pkl",
                   ckpt_dir="/kaggle/input/casia-b-opengait/checkpoints"),
    # Colab: Drive mirror of the same Dataset.
    "colab": dict(repo="/content/cross_attack_gait",
                  opengait="/content/OpenGait",
                  dataset_root="/content/drive/MyDrive/gait/CASIA-B-pkl",
                  ckpt_dir="/content/drive/MyDrive/gait/checkpoints"),
    # Local: set CASIAB_PKL and GAIT_CKPT_DIR (e.g. ../data/CASIA-B-pkl). CPU-only machines cannot run
    # OpenGait models (BaseModel hardcodes CUDA); official gate runs on GPU.
    "local": dict(repo=str(Path.cwd().parent), opengait=str(Path.cwd().parent.parent / "OpenGait"),
                  dataset_root=os.environ.get("CASIAB_PKL", ""), ckpt_dir=os.environ.get("GAIT_CKPT_DIR", "")),
}[PLATFORM]
TEAM_REPO_URL = "https://github.com/haragam22/gait-paper.git"   # e.g. https://github.com/<org>/cross_attack_gait.git (private; use a token on Kaggle)

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
if PLATFORM != "local" and not Path(PATHS["repo"]).exists():
    env = dict(os.environ)
    if PLATFORM == "kaggle":  # private repo: add a GitHub token as Kaggle secret GITHUB_TOKEN (Add-ons > Secrets)
        import base64
        from kaggle_secrets import UserSecretsClient
        auth = base64.b64encode(f"x-access-token:{UserSecretsClient().get_secret('GITHUB_TOKEN')}".encode()).decode()
        # token via env-only git config: never in argv, error messages, or .git/config (saved with notebook output)
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    subprocess.run(["git", "clone", TEAM_REPO_URL, PATHS["repo"]], check=True, env=env)
sys.path.insert(0, PATHS["repo"]); os.chdir(PATHS["repo"])

import src.utils as U
BASE = U.load_yaml("configs/baseline.yaml")
if not Path(PATHS["opengait"]).exists():
    subprocess.run(["git", "clone", "https://github.com/ShiqiYu/OpenGait.git", PATHS["opengait"]], check=True)
subprocess.run(["git", "-C", PATHS["opengait"], "checkout", "-q", BASE["opengait_commit"]], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "tensorboard"], check=True)

# Kaggle data: use a packed CASIA-B-pkl attached as input (team Dataset or this notebook's saved output);
# otherwise pack the public PNG mirror (trnquanghuyn/casia-b, attach as input) with OpenGait's own script,
# same command as docs/reproducibility.md (~20 min). "Save Version" keeps /kaggle/working/CASIA-B-pkl so
# teammates attach this notebook's output instead of re-packing.
if PLATFORM == "kaggle" and not Path(PATHS["dataset_root"]).exists():
    # Attached inputs mount at /kaggle/input/<slug>/ or deeper (datasets/<owner>/<slug>/, notebooks/...).
    find_input = lambda rel: (p for pat in ("*/", "*/*/", "*/*/*/", "*/*/*/*/") for p in Path("/kaggle/input").glob(pat + rel))
    attached = next(find_input("CASIA-B-pkl"), None)
    packed = Path("/kaggle/working/CASIA-B-pkl")
    if attached is None and not (packed / ".done").exists():
        raw = next(find_input("output/001"), None)
        if raw is None:
            tree = sorted(str(p) for pat in ("*", "*/*", "*/*/*") for p in Path("/kaggle/input").glob(pat) if p.is_dir())
            raise FileNotFoundError("No CASIA-B-pkl and no raw CASIA-B (output/001) under /kaggle/input. Attach "
                                    "Notebook 01's saved output (Add Input > Your Work) or trnquanghuyn/casia-b. "
                                    "Found:\n" + "\n".join(tree[:40]))
        raw = raw.parent
        subprocess.run([sys.executable, "datasets/pretreatment.py", "-i", str(raw), "-o", str(packed),
                        "-d", "GREW", "-n", str(os.cpu_count())], cwd=PATHS["opengait"], check=True)
        (packed / ".done").touch()
    PATHS["dataset_root"] = str(attached or packed)
print("dataset_root:", PATHS["dataset_root"])
BASE["opengait_dir"] = PATHS["opengait"]
BASE["dataset_root"] = PATHS["dataset_root"]

In [ ]:
# --- Cell 1 record: date, seed, versions, GPU, platform, OpenGait commit --------------
SEED = BASE["seed"]
U.seed_everything(SEED)
ENV = U.env_record(SEED, BASE["opengait_dir"], {"notebook": "04_temporal_masked_pgd"})
assert ENV["opengait_commit"] == BASE["opengait_commit"], ENV["opengait_commit"]
ATK = U.load_yaml("configs/attacks.yaml")
ENV

In [ ]:
# --- Checkpoint: from the team Dataset, else download the released one -----------------
import urllib.request
gb = BASE["models"]["gaitbase"]
ck = Path(PATHS["ckpt_dir"]) / "GaitBase_DA-60000.pt"
if not ck.exists():
    ck = Path("/kaggle/working" if PLATFORM == "kaggle" else ".") / "GaitBase_DA-60000.pt"
    if not ck.exists():
        urllib.request.urlretrieve(gb["checkpoint_url"], ck)
gb["checkpoint"] = str(ck)
print("GaitBase:", ck, U.file_sha256(ck)[:12])

In [ ]:
# --- Load GaitBase + clean gallery ---------------------------------------------------
import time, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from src.evaluation import load_model, Gallery, evaluate_attack, AttackTarget, read_results, summarize_runs
from src.data import stratified_subset, save_subset, load_subset
from src.attacks import masks as M
from src.attacks.pgd import PGD, dense, edge, temporal, margin_grad, run_tier_a
model, data, og_cfg = load_model(BASE, "gaitbase")
gallery = Gallery.build(model, data, cache=f"results/processed/gallery_{model.name}_{model.checkpoint_id}.npz")
ENV.update(checkpoint=gb["checkpoint"], checkpoint_id=model.checkpoint_id)
print(model.name, model.checkpoint_id, model.device, "gallery", tuple(gallery.emb.shape))

def push(msg):
    pushed = U.push_results(PATHS["repo"], msg, U.github_token(PLATFORM), paths=("results", "configs"))
    print("pushed" if pushed else "nothing to push")

In [ ]:
# The fixed attack subset from Notebook 02 — identical probes for every attack (CLAUDE.md rule 7).
SUBSET = Path("results/processed/subset_attack.csv")
assert SUBSET.exists(), "run Notebook 02 first (it writes and pushes the subset); then git pull / re-clone"
subset = load_subset(SUBSET, data)
print(len(subset), "probes"); pd.Series([r.condition for r in subset]).value_counts()

## 1. Frame selection on one probe (sanity check)

In [ ]:
T_CFG = ATK["families"]["temporal"]
rec = subset[0]; x = data.load(rec).to(model.device); tgt = AttackTarget.for_probe(gallery, rec)
_, g = margin_grad(model, x, tgt)
score = g[0].abs().flatten(1).sum(1).cpu().numpy()
fig, ax = plt.subplots(figsize=(8, 2.5)); ax.bar(range(len(score)), score)
for k in T_CFG["k_frames"]:
    print(f"k={k}: random_k {M.random_frames(x.shape[1], k, M.sample_seed(SEED, rec.sample_id))} | "
          f"topk_grad {M.topk_grad_frames(g, k)}")
ax.set_xlabel("frame"); ax.set_ylabel("|grad| (L1)"); ax.set_title(f"{rec.sample_id}: per-frame gradient at clean input")
plt.tight_layout()

## 2. Sweep: (rule, k) at ε = 8/255 first, then the rest of the ε grid

In [ ]:
OUT = Path("results/raw/temporal_masked_results.csv")
grid = [(r, k) for r in T_CFG["selection"] for k in T_CFG["k_frames"]]
order = [(r, k, 8) for r, k in grid] + [(r, k, e) for e in ATK["eps_grid"] if e != 8 for r, k in grid]
for rule, k, e255 in order:
    t0 = time.time()
    df = run_tier_a(model, data, gallery, subset, f"temporal_{rule}_k{k}", temporal(k, rule), e255, ATK, SEED,
                    OUT, SUBSET, run_meta={"env": ENV, "rule": rule, "k": k})
    print(f"{rule} k={k} eps {e255}/255: success {df.attack_success.mean():.3f}, {(time.time()-t0)/60:.1f} min")
    push(f"results: Notebook 04 temporal PGD {rule} k={k} eps={e255}/255 ({ENV['platform']})")

## 3. Summary

In [ ]:
R = read_results("results/raw/temporal_masked_results.csv")
R = R[R.run_id.str.endswith(model.checkpoint_id) & R.sample_id.isin([s.sample_id for s in subset])]
display(summarize_runs(R, ["attack", "eps_255"]).round(3))
fig, ax = plt.subplots(figsize=(5.5, 3.5))
R = R.assign(eps_255=(R.eps * 255).round(2))
for c, g in R.groupby("attack"):
    s = g.groupby("eps_255").adv_rank1_viewavg.mean() * 100
    ax.plot(s.index, s.values, marker="o", label=c)
ax.set_xlabel("ε × 255"); ax.set_ylabel("adversarial Rank-1 (%)"); ax.set_xscale("log", base=2)
ax.set_title("Temporal-sparse PGD"); ax.legend(fontsize=7)
plt.tight_layout(); Path("results/figures").mkdir(exist_ok=True); plt.savefig("results/figures/04_pgd_temporal_rank1.png", dpi=150)
push("results: Temporal-sparse PGD summary figure")

In [ ]:
# random_k vs topk_grad at equal k: does choosing frames by gradient matter?
S = summarize_runs(R, ["attack", "eps_255", "condition"]).reset_index()
display(S[S.eps_255 == 8].pivot(index="attack", columns="condition", values="attack_success").round(3))

## Findings, limitations, next step
*(fill after running — copy numbers from the summary table, do not retype them)*

- random_k vs topk_grad at equal k:
- Effect of k:
- Frames perturbed (check = k):
- Oct 23 checkpoint: pgd / edge_masked / temporal_masked CSVs on identical sample_ids?
- Next: Notebook 07 (Hou et al.).